<h1>COMP90049 Project

Author: Chawanrat Wisitphongphiboon \
Student ID: 1603089

1. Feature construction and Preprocessing data

- Feature construction <br>
New features are added to the dataset based on existing columns.  <br>
These include averages of pollutants, combinations of values (like PM2.5 and PM10), and health-related scores.  <br>
These extra features help the model find patterns and improve predictions.

In [9]:
import pandas as pd
import numpy as np

# Load CSV
df = pd.read_csv("air_quality_health_impact_data.csv")

# Pollutant Ratios and Aggregations
df['PM2_5_to_PM10'] = df['PM2_5'] / df['PM10']
df['NO2_to_SO2'] = df['NO2'] / (df['SO2'] + 1e-5)
df['O3_to_NO2'] = df['O3'] / (df['NO2'] + 1e-5)
df['PollutantTotal'] = df[['PM10', 'PM2_5', 'NO2', 'SO2', 'O3']].sum(axis=1)
df['PollutantMean'] = df[['PM10', 'PM2_5', 'NO2', 'SO2', 'O3']].mean(axis=1)
df['PM_diff'] = df['PM10'] - df['PM2_5']
df['O3_NO2_SO2_ratio'] = df['O3'] / (df['NO2'] + df['SO2'] + 1e-5)

# Weather Interactions
df['HeatIndex'] = df['Temperature'] * df['Humidity'] / 100
df['Wind_Pollutant_Effect'] = df['WindSpeed'] * df['PollutantTotal']
df['Temp_Humidity'] = df['Temperature'] * df['Humidity']
df['Humidity_Wind'] = df['Humidity'] * df['WindSpeed']
df['Combined_Weather_Index'] = df['Temperature'] + df['Humidity'] + df['WindSpeed']
df['Weather_Variability'] = df[['Temperature', 'Humidity', 'WindSpeed']].std(axis=1)

# Health Ratios
df['RespiratoryPerAQI'] = df['RespiratoryCases'] / (df['AQI'] + 1e-5)
df['CardioPerPM10'] = df['CardiovascularCases'] / (df['PM10'] + 1e-5)
df['HospitalPerO3'] = df['HospitalAdmissions'] / (df['O3'] + 1e-5)
df['CardioResp_Ratio'] = df['CardiovascularCases'] / (df['RespiratoryCases'] + 1e-5)

# Category Binning
df['TempLevel'] = pd.cut(df['Temperature'], bins=[-np.inf, 10, 25, np.inf], labels=['Low', 'Medium', 'High'])
df['HumidityLevel'] = pd.cut(df['Humidity'], bins=[-np.inf, 30, 70, np.inf], labels=['Low', 'Medium', 'High'])
df['AQI_Category'] = pd.cut(df['AQI'], bins=[-np.inf, 50, 100, 150, 200, np.inf], labels=['Good', 'Moderate', 'Unhealthy', 'Very Unhealthy', 'Hazardous'])

# Polynomial and Log Features
df['PM10_sq'] = df['PM10'] ** 2
df['log_PM2_5'] = np.log1p(df['PM2_5'])
df['AQI_sq'] = df['AQI'] ** 2
df['log_NO2'] = np.log1p(df['NO2'])
df['log_SO2'] = np.log1p(df['SO2'])

# Simulated Temporal Features
df['DayOfWeek'] = df['RecordID'] % 7
df['isWeekend'] = df['DayOfWeek'].isin([0, 6]).astype(int)

# One-hot Encoding
df = pd.get_dummies(df, columns=['TempLevel', 'HumidityLevel', 'AQI_Category'], drop_first=True)

# Additional Interactions & Polynomials
df['AQI_Temp'] = df['AQI'] * df['Temperature']
df['AQI_WindSpeed'] = df['AQI'] * df['WindSpeed']
df['Temp_sq'] = df['Temperature'] ** 2
df['Humidity_sq'] = df['Humidity'] ** 2
df['WindSpeed_sq'] = df['WindSpeed'] ** 2


In [10]:
# Check number of rows and columns
print("Shape of dataset (rows, columns):", df.shape)

# Check for missing values
print("\nMissing values per column:")
print(df.isnull().sum())

# Check data types
print("\nData types:")
print(df.dtypes)

# Check class distribution
print("\nClass distribution in HealthImpactClass:")
print(df['HealthImpactClass'].value_counts())

Shape of dataset (rows, columns): (5811, 52)

Missing values per column:
RecordID                       0
AQI                            0
PM10                           0
PM2_5                          0
NO2                            0
SO2                            0
O3                             0
Temperature                    0
Humidity                       0
WindSpeed                      0
RespiratoryCases               0
CardiovascularCases            0
HospitalAdmissions             0
HealthImpactScore              0
HealthImpactClass              0
PM2_5_to_PM10                  0
NO2_to_SO2                     0
O3_to_NO2                      0
PollutantTotal                 0
PollutantMean                  0
PM_diff                        0
O3_NO2_SO2_ratio               0
HeatIndex                      0
Wind_Pollutant_Effect          0
Temp_Humidity                  0
Humidity_Wind                  0
Combined_Weather_Index         0
Weather_Variability            0
Res

- Preprocessing data

In [11]:
# Drop the regression target to avoid leakage
df = df.drop(columns=['HealthImpactScore'])

# Separate features (X) and target (y)
X = df.drop(columns=['HealthImpactClass'])
y = df['HealthImpactClass']

2. Apply machine learning algorithms <br>
Three machine learning models are used to predict the Health Impact Class:<br>
    1. Logistic Regression<br>
    2. Decision Tree Classifier<br>
    3. Random Forest Classifier<br><br>

    Each model is tested using cross-validation to get accurate results and find the best settings.<br>
    Model performance is measured using accuracy, precision, recall, and F1-score.

In [12]:
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Outer and inner folds
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Scoring metrics
scoring = {
    'accuracy': 'accuracy',
    'precision_macro': 'precision_macro',
    'recall_macro': 'recall_macro',
    'f1_macro': 'f1_macro'
}

# Store results
results = {}

# Logistic Regression
logreg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
])
logreg_params = {'clf__C': [0.1, 1, 10]}
logreg_grid = GridSearchCV(logreg_pipeline, logreg_params, cv=inner_cv, scoring='f1_macro')
logreg_result = cross_validate(logreg_grid, X, y, cv=outer_cv, scoring=scoring)
results['Logistic Regression'] = {k: v.mean() for k, v in logreg_result.items() if k.startswith('test_')}

# Decision Tree
from sklearn.tree import DecisionTreeClassifier
dt_grid = GridSearchCV(DecisionTreeClassifier(class_weight='balanced', random_state=42),
                       param_grid={'max_depth': [5, 10, 15]}, cv=inner_cv, scoring='f1_macro')
dt_result = cross_validate(dt_grid, X, y, cv=outer_cv, scoring=scoring)
results['Decision Tree'] = {k: v.mean() for k, v in dt_result.items() if k.startswith('test_')}

# Random Forest
from sklearn.ensemble import RandomForestClassifier
rf_grid = GridSearchCV(RandomForestClassifier(class_weight='balanced', random_state=42),
                       param_grid={'n_estimators': [50, 100, 200]}, cv=inner_cv, scoring='f1_macro')
rf_result = cross_validate(rf_grid, X, y, cv=outer_cv, scoring=scoring)
results['Random Forest'] = {k: v.mean() for k, v in rf_result.items() if k.startswith('test_')}

# Display nicely
results_df = pd.DataFrame(results).T
print(results_df.round(4))


c:\Users\chawa\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\chawa\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\chawa\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\chawa\anaconda3\Lib\site-packag

                     test_accuracy  test_precision_macro  test_recall_macro  \
Logistic Regression         0.7169                0.3682             0.5270   
Decision Tree               0.8931                0.5402             0.5944   
Random Forest               0.9184                0.6654             0.4836   

                     test_f1_macro  
Logistic Regression         0.3923  
Decision Tree               0.5571  
Random Forest               0.5305  
